# Configuration

In [1]:
import os 

if True ^ os.getcwd().endswith('hte-and-targeting'):
    os.chdir('..')

In [2]:
import pandas as pd 
import numpy as np

In [3]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['text.usetex'] = False

In [4]:
from statsmodels.regression.linear_model import OLS

In [5]:
from core.variables import * 
from core.segment_targeting_estimators import *
from core.dgp import *
from core.segment_targeting_experiments import *
from core.visualization import *

# DGP

Demand model via potential outcome: 
$$
Y_i = \tau_{g(X_i)}T_i + \epsilon_i
$$
where $g: \mathcal{X} \mapsto \mathcal{S}$ denote the segmentation function and $\mathcal{S}$ is the set of discrete segments. 
The expected potential outcome depends on the individual characteristics only through segmentation. 
The following table summarizes the expected potential outcome: 
|           | Treatment 1 | Treatment 2 | ... | Treatment $T$ | Control |
|-----------|:-------------:|:-------------:|:-----:|:-------------:|:-------------:|
| Segment 1 | $a_{11}$    | $a_{12}$      | ... | $a_{1T}$      | $a_{10}$      |
| Segment 2 | $a_{21}$    | $a_{22}$      | ... | $a_{2T}$      | $Y_{20}$      |
| $\vdots$  | $\vdots$    | $\vdots$      | $\ddots$ | $\vdots$      | $\vdots$      |
| Segment $S$ | $a_{S1}$    | $a_{S2}$      | ... | $a_{ST}$      | $a_{S0}$      |

Consumer characteristics are assumed to be standard normal:  
$$
X_i \sim \text{i.i.d. } \mathcal{N}(0, 1)
$$

In [247]:
class MultipleTreatmentsMultipleSegments(object):
    def __init__(
        self, n_segments: int, te_diff: float, segment_func: callable, 
        treatment_space: np.ndarray, noise_std: float = 1.0, 
        seed: int = None, **kwargs
    ):
        if seed is not None:
            np.random.seed(seed)

        self.n_segments = n_segments
        self.te_diff = te_diff
        self.segment_te_arr = np.array([(i + 1) * self.te_diff for i in range(self.n_segments)])
        self.segment_func = segment_func
        self.noise_std = noise_std
        self.treatment_space = treatment_space

    def sample_individuals(self, sample_size: int, seed: int = None) -> np.ndarray:
        """ 
        Sample individuals from the DGP. 
        """
        if seed is not None:
            np.random.seed(seed)
        return np.random.normal(0, 1, size=(sample_size, ))
    
    def sample(self, sample_size, seed: int = None) -> pd.DataFrame:
        if seed is not None:
            np.random.seed(seed)

        # generate individuals for traning data
        covariates = self.sample_individuals(sample_size, seed=seed)  # shape = (sample_size, )
        segments = self.segment_func(covariates)  # shape = (sample_size, )
        consumer_te_arr = self.segment_te_arr[segments]  # shape = (sample_size, )

        # randomly assign treatment
        treatments = np.random.choice(self.treatment_space, size=sample_size)

        # calculate outcomes
        epsilon = np.random.normal(0, self.noise_std, size=sample_size)
        outcomes = consumer_te_arr * treatments + epsilon

        return pd.DataFrame({
            'outcome': outcomes, 'treatment': treatments, 'covariates': covariates, 
        })
    
    @property
    def expected_potential_outcome_table(self) -> pd.DataFrame:
        """
        Generate the expected potential outcome table. 
        """
        return pd.DataFrame.from_records([
            {'segment': segment, 'treatment': treatment, 'expected_outcome': treatment * self.segment_te_arr[segment]}
            for segment in range(self.n_segments) for treatment in self.treatment_space
        ]).pivot(index='segment', columns='treatment', values='expected_outcome')
    
    @property
    def treatment_effect_table(self) -> pd.DataFrame:
        """
        Generate the treatment effect table. 
        """
        return self.expected_potential_outcome_table.apply(lambda x: x - x[0], axis=1).drop(0, axis=1)

# Demand Model

In [207]:
class PotentialOutcomeModel(object):
    def __init__(self, n_segments: int, segment_func: callable, treatment_space: np.ndarray) -> None:
        self.segment_func = segment_func 
        self.treatment_space = treatment_space
        self.n_segments = n_segments 

        # placeholder
        self.avg_outcome_table = None
        self.se_outcome_table = None

    def fit(self, data):
        data = data.copy()

        # add segments
        data['segment'] = self.segment_func(data['covariates'])
        
        # calculate the average outcomes
        self.avg_outcome_table = data.groupby(['segment', 'treatment'])['outcome'].mean().unstack()

        # calculate the standard errors
        data = data.groupby(['segment', 'treatment'])['outcome'].agg(['count', 'std'])
        data['se'] = data['std'] / np.sqrt(data['count'])

        self.se_outcome_table = data.drop(['count', 'std'], axis=1).unstack()
                
    @staticmethod
    def difference_in_mean(treatment_val: int, treatments: np.ndarray, outcomes: np.ndarray) -> float:
        return outcomes[treatments == treatment_val].mean() - outcomes[treatments == 0].mean()
    
    @property
    def treatment_effect_table(self) -> pd.DataFrame:
        if self.avg_outcome_table is None:
            raise ValueError('Model has not been fitted yet.')
        
        return self.avg_outcome_table.apply(lambda x: x - x[0], axis=1).drop(0, axis=1)

# Optimization

In [262]:
def optimize(
    test_customers: np.ndarray, treatment_effect_table: pd.DataFrame, 
    price: float, cost: float, segment_func: callable
) -> dict:
    """
    Optimize the plugin problem. 
    """
    # make sure the control group is not included in the treatment effect table
    assert 0 not in treatment_effect_table.columns, 'Control group should not be included in the treatment effect table.'

    # get the segment for each customer
    customer_segment_arr = segment_func(test_customers)  # shape = (sample_size, )

    # get the treatment effect for each customer
    customer_te_arr = treatment_effect_table.values[customer_segment_arr, :]  # shape = (sample_size, treatment_space - 1)
    
    # add the control group
    customer_te_arr = np.insert(customer_te_arr, 0, 0, axis=1)  # shape = (sample_size, treatment_space)

    # calculate the profit lift for each customer and each treatment
    profit_lift_arr = customer_te_arr * price - dgp.treatment_space.reshape(1, -1) * cost

    # optimize
    opt_targ_decision = profit_lift_arr.argmax(axis=1)
    opt_targ_val_est = profit_lift_arr.max(axis=1).sum()

    return {
        'decision': opt_targ_decision, 'obj_val': opt_targ_val_est
    }

def objective_func(
        test_customers: np.ndarray, targ_decision: np.ndarray,
        treatment_effect_table: pd.DataFrame,
        price: float, cost: float, segment_func: callable, 
) -> float:
    """
    Objective function for the plugin problem. 
    """
    # make sure the control group is not included in the treatment effect table
    assert 0 not in treatment_effect_table.columns, 'Control group should not be included in the treatment effect table.'

    # get the segment for each customer
    customer_segment_arr = segment_func(test_customers)  # shape = (sample_size, )

    # get the treatment effect for each customer
    customer_te_arr = treatment_effect_table.values[customer_segment_arr, :]  # shape = (sample_size, treatment_space - 1)
    
    # add the control group
    customer_te_arr = np.insert(customer_te_arr, 0, 0, axis=1)  # shape = (sample_size, treatment_space)

    # calculate the profit lift for each customer and each treatment
    profit_lift_arr = customer_te_arr * price - dgp.treatment_space.reshape(1, -1) * cost

    # calculate the objective value
    obj_val = profit_lift_arr[np.arange(profit_lift_arr.shape[0]), targ_decision].sum()

    return obj_val

# Winner's Curse

In [ ]:
def repeated_experiments(
    targeting_params: dict, 
    demand_params: dict, 
    data_params: dict, 
    experiment_params: dict, 
    estimators_dict: dict, 
) -> list:
    pass 

In [ ]:
dgp = MultipleTreatmentsMultipleSegments(
    n_segments=2, te_diff=0.05, segment_func=lambda x: (x >= 0).astype(int), 
    treatment_space=np.array([0, 1, 2, 3]), noise_std=2.0, seed=0
)

In [ ]:
targeting_params = {'price': 1.5, 'cost': 0.1}
demand_params = {
    'n_segments': 2, 
    'segment_func': lambda x: (x >= 0).astype(int),
    'treatment_space': np.array([0, 1, 2, 3]), 
    'noise_std': 1.0, 
}
data_params = {'sample_size': 1000}
experiment_params = {'n_experiments': 1000}

estimators_dict = {
    'plugin': {
        'estimator': optimize, 
        'params': {}
    }
}